In [10]:
from google.colab import drive
import os
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [11]:
cd /content/gdrive/MyDrive/project

/content/gdrive/.shortcut-targets-by-id/1IhtrClOvYkKUSZxwUoLOhxwHY9CdXCaH/project


In [12]:
PROJECT_PATH = '/content/drive/MyDrive/project'
print(f"Current Working Directory: {os.getcwd()}")
!pip install efficientnet_pytorch

Current Working Directory: /content/gdrive/.shortcut-targets-by-id/1IhtrClOvYkKUSZxwUoLOhxwHY9CdXCaH/project


upload files

In [13]:
%%writefile code/common.py
# --- CONSTANTS ---
MAX_LEN = 5
NUM_CLASS = 10   # Digits 0-9
PADDING_TOKEN = 10  # Class index 10 is 'Blank/Padding'

Overwriting code/common.py


In [14]:
%%writefile code/train/dataset.py
import json
import os
import zipfile
import io
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import torch
import sys

# Allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from common import MAX_LEN, PADDING_TOKEN

class MCharDataset(Dataset):
    def __init__(self, path, json_file, transform=None, mode='train'): # new bool var mode for distinguish purposes on augmentation

        self.path = path
        if transform is not None:

          self.transform = transform
        else:
          normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # normalise the image

          if mode == 'train':
              self.transform = transforms.Compose([
                  transforms.Resize((128, 128)), # resize to fixed size

                    # 1. Geometric random rotate (-10 to 10 degrees)
                  transforms.RandomRotation(10),

                    # 2. Geometric random shifting
                  transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)),

                    # 3. Color and lighting changes (Day/Night/Shadows)
                  transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),

                    # 4. blury
                  transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),

                  transforms.ToTensor(),
                  normalize
                ])
          else:
                # only resize and normalize
              self.transform = transforms.Compose([
                  transforms.Resize((128, 128)),
                  transforms.ToTensor(),
                  normalize
                ])

        # Load labels
        with open(json_file, "r") as f:
            self.meta = json.load(f)
        self.keys = list(self.meta.keys())

        # Detect if input is a ZIP file
        self.is_zip = os.path.isfile(path) and (path.endswith('.zip') or zipfile.is_zipfile(path))

        # Zip handling variables
        self.zip_handle = None
        self.prefix = ""

        if self.is_zip:
            # Figuring out if images are in a subfolder inside the zip
            # e.g., mchar_train.zip -> mchar_train/000000.png
            with zipfile.ZipFile(path, 'r') as zf:
                names = zf.namelist()
                sample_key = self.keys[0] # e.g. "000000.png"

                if sample_key in names:
                    self.prefix = ""
                else:
                    # Check common prefix like "mchar_train/"
                    folder_name = os.path.basename(path).replace('.zip', '')
                    if f"{folder_name}/{sample_key}" in names:
                        self.prefix = f"{folder_name}/"
                    else:
                        # Fallback search
                        for n in names[:100]:
                            if n.endswith(sample_key):
                                self.prefix = n[:-len(sample_key)]
                                break

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        name = self.keys[idx] # e.g., "000000.png"

        if self.is_zip:
            # Open zip in the worker process (thread-safe way)
            if self.zip_handle is None:
                self.zip_handle = zipfile.ZipFile(self.path, 'r')

            try:
                full_path = self.prefix + name
                img_bytes = self.zip_handle.read(full_path)
                img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            except Exception:
                # Return black image if file is corrupt/missing
                img = Image.new('RGB', (128, 64))
        else:
            # Standard folder reading
            img_path = os.path.join(self.path, name)
            try:
                img = Image.open(img_path).convert("RGB")
            except:
                img = Image.new('RGB', (128, 64))

        if self.transform:
            img = self.transform(img)

        labels = self.meta[name]["label"]
        # Pad label with 10
        if len(labels) < MAX_LEN:
            labels = labels + [PADDING_TOKEN] * (MAX_LEN - len(labels))

        return img, torch.tensor(labels[:MAX_LEN], dtype=torch.long)

Overwriting code/train/dataset.py


In [15]:
%%writefile code/train/train.py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import sys
import os

# --- PATH SETUP ---
current_dir = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, "..", ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "code"))

# --- IMPORTS ---
from dataset import MCharDataset
from model import OCRModel

def train():
    # --- CONFIG ---
    EPOCHS = 60           # 60 is usually enough with EfficientNet
    BATCH_SIZE = 128      # Increased batch size (since we lowered image size slightly)
    LR = 1e-3

    # NEW: Optimized Image Size (Height=64, Width=128)
    # House numbers are wide. 64x128 is a very standard OCR resolution.
    IMG_H, IMG_W = 64, 128

    # Paths
    tcdata_dir = os.path.join(PROJECT_ROOT, "tcdata")
    output_dir = os.path.join(PROJECT_ROOT, "user_data", "model_data")
    os.makedirs(output_dir, exist_ok=True)

    resume_path = os.path.join(output_dir, "latest.pth")
    best_model_path = os.path.join(output_dir, "model.dat")

    # --- TRANSFORMS ---
    # We use a slightly more aggressive augmentation
    train_tf = transforms.Compose([
        transforms.Resize((IMG_H, IMG_W)),
        transforms.RandomAffine(
            degrees=10,
            translate=(0.1, 0.1),
            scale=(0.8, 1.1),
            shear=5  # Added shear
        ),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    val_tf = transforms.Compose([
        transforms.Resize((IMG_H, IMG_W)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # --- DATASET LOADING ---
    train_img = os.path.join(tcdata_dir, "mchar_train")
    if not os.path.exists(train_img): train_img += ".zip"

    val_img = os.path.join(tcdata_dir, "mchar_val")
    if not os.path.exists(val_img): val_img += ".zip"

    train_json = os.path.join(tcdata_dir, "mchar_train.json")
    val_json = os.path.join(tcdata_dir, "mchar_val.json")

    if not os.path.exists(train_json):
        print(f"Dataset JSON not found at {train_json}")
        return

    train_set = MCharDataset(train_img, train_json, transform=train_tf)
    val_set = MCharDataset(val_img, val_json, transform=val_tf)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=2)

    # --- MODEL SETUP ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = OCRModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    # Scheduler: Drop LR if validation loss stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    # NEW: Label Smoothing
    # This helps the model generalize better by not forcing 100% confidence
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    # --- RESUME LOGIC ---
    start_epoch = 0
    best_loss = float('inf')

    if os.path.exists(resume_path):
        print(f"Resuming from {resume_path}...")
        checkpoint = torch.load(resume_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_loss = checkpoint.get('best_loss', float('inf'))
    else:
        print("Starting training from scratch.")

    # --- TRAINING LOOP ---
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        total_loss = 0

        for i, (img, label) in enumerate(train_loader):
            img, label = img.to(device), label.to(device)

            optimizer.zero_grad()

            # Forward
            out = model(img) # list of 5 tensors
            out_stack = torch.stack(out, dim=1) # [Batch, 5, 11]

            # Compute loss
            # Permute to [Batch, Classes, Seq_Len] for CrossEntropy
            loss = loss_fn(out_stack.permute(0, 2, 1), label)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if i % 100 == 0:
                print(f"\rEpoch {epoch} [{i}/{len(train_loader)}] Loss: {loss.item():.4f}", end="")

        avg_loss = total_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"\nEpoch {epoch} Avg Train Loss: {avg_loss:.4f} | LR: {current_lr:.1e}")

        # --- VALIDATION ---
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for img, label in val_loader:
                img, label = img.to(device), label.to(device)

                out = model(img)
                out_stack = torch.stack(out, dim=1)

                loss = loss_fn(out_stack.permute(0, 2, 1), label)
                val_loss += loss.item()

                # Calculate simple accuracy (perfect match of all digits)
                # preds: [Batch, 5]
                preds = out_stack.argmax(dim=2)

                # Check row by row equality
                # eq: [Batch, 5] -> all(dim=1) -> [Batch] boolean
                row_match = torch.all(preds.eq(label), dim=1)
                correct += row_match.sum().item()
                total += label.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / total
        print(f"Validation Loss: {avg_val_loss:.4f} | Acc: {val_acc:.4f}")

        scheduler.step(avg_val_loss)

        # --- SAVE CHECKPOINTS ---
        # We save if validation loss improves
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            print(f">>> Found Better Model (Loss: {best_loss:.4f}). Saving...")
            torch.save(model.state_dict(), best_model_path)

        # Always save latest for resuming
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss': best_loss
        }, resume_path)

if __name__ == "__main__":
    train()

Overwriting code/train/train.py


In [16]:
%%writefile code/train/model.py
import torch.nn as nn
import torchvision.models as models
import sys
import os

# Allow import from parent directory
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from common import MAX_LEN, NUM_CLASS

class OCRModel(nn.Module):
    def __init__(self):
        super().__init__()

        # 1. Load Pretrained EfficientNet
        try:
            # Newer Torchvision versions
            weights = models.EfficientNet_B0_Weights.DEFAULT
            base = models.efficientnet_b0(weights=weights)
        except:
            # Older Torchvision versions fallback
            base = models.efficientnet_b0(pretrained=True)

        # EfficientNet B0 final feature map size is 1280
        self.backbone = base

        # 2. IMPROVEMENT: Bigger Bottleneck + BatchNorm
        # We increase hidden size to 1024 (was 256) to keep more features
        hidden_size = 1024

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),                 # Increased dropout slightly
            nn.Linear(1280, hidden_size),
            nn.BatchNorm1d(hidden_size),     # NEW: Stabilizes training
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

        # 3. Prediction Heads (5 heads for 5 characters)
        # Each head outputs 11 classes (0-9 + Padding)
        self.heads = nn.ModuleList([
            nn.Linear(hidden_size, NUM_CLASS + 1) for _ in range(MAX_LEN)
        ])

    def forward(self, x):
        # Extract features
        feat = self.backbone(x)

        # Pass through each head
        out = [h(feat) for h in self.heads]
        return out

Overwriting code/train/model.py


In [17]:
%%writefile code/test/test.py
import torch
from torch.utils.data import DataLoader
import sys
import os

# Setup Paths
current_dir = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, "..", ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "code"))

from train.model import OCRModel
from common import PADDING_TOKEN
from dataset_test import ZipTestDataset

def predict():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Paths
    model_path = os.path.join(PROJECT_ROOT, "user_data", "model_data", "model.dat")

    possible_zips = [
        os.path.join(PROJECT_ROOT, "tcdata", "mchar_test_a.zip"),
        os.path.join(PROJECT_ROOT, "tcdata", "test_a.zip")
    ]

    zip_file = None
    for z in possible_zips:
        if os.path.exists(z):
            zip_file = z
            break

    if zip_file is None:
        print("Error: Could not find mchar_test_a.zip")
        return

    # Load Model
    model = OCRModel().to(device)
    if not os.path.exists(model_path):
        print("Error: Model file not found. Train first.")
        return

    checkpoint = torch.load(model_path, map_location=device)

    # Flexible loading (handle if saved as full dict or just weights)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)

    model.eval()

    # Dataset
    test_set = ZipTestDataset(zip_file)
    loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)

    results = []
    print(f"Inference on {len(test_set)} images...")

    with torch.no_grad():
        for i, (img, names) in enumerate(loader):
            img = img.to(device)

            # Forward
            out = model(img) # List of 5 tensors
            out_stack = torch.stack(out, dim=1) # [Batch, 5, 11]

            # Greedy Decode
            preds = out_stack.argmax(dim=2).cpu().numpy()

            for filename, seq in zip(names, preds):
                # Filter padding (token 10)
                pred_str = "".join([str(x) for x in seq if x != PADDING_TOKEN])
                results.append(f"{filename},{pred_str}")

            if i % 20 == 0:
                print(f"\rBatch {i}/{len(loader)}", end="")

    # Save
    out_file = os.path.join(PROJECT_ROOT, "prediction_result", "result.csv")
    os.makedirs(os.path.dirname(out_file), exist_ok=True)

    with open(out_file, "w") as f:
        f.write("file_name,file_code\n")
        f.write("\n".join(results))

    print(f"\nDone. Saved to {out_file}")

if __name__ == "__main__":
    predict()

Overwriting code/test/test.py


In [18]:
%%writefile code/test/dataset_test.py
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import zipfile
import io

class ZipTestDataset(Dataset):
    def __init__(self, zip_path):
        self.zip_path = zip_path

        # Open zip once to get the list of files
        with zipfile.ZipFile(zip_path, 'r') as zf:
            # Filter only images
            self.file_list = [
                f for f in zf.namelist()
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
                and not f.startswith('__MACOSX')
            ]

        # Sort to ensure consistent order
        self.file_list = sorted(self.file_list)

        # NEW: Match the training resolution (Height=64, Width=128)
        self.tf = transforms.Compose([
            transforms.Resize((64, 128)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        self.zip_handle = None

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        if self.zip_handle is None:
            self.zip_handle = zipfile.ZipFile(self.zip_path, 'r')

        file_path_in_zip = self.file_list[idx]

        # Read bytes
        img_bytes = self.zip_handle.read(file_path_in_zip)
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        # Apply transforms
        img = self.tf(img)

        # clean filename
        clean_name = os.path.basename(file_path_in_zip)

        return img, clean_name

Overwriting code/test/dataset_test.py
